In [ ]:
# LSTM으로 분류 모델 작성(감성분류)
docs = ['너무 재밌네요','최고예요','참 잘 만든 영화예요','추천하고 싶은 영화입니다','한 번 더 보고 싶네요',
        '글쎄요','별로네요','생각보다 지루하네요','연기가 어색해요','재미없어요']

import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer

labels = np.array([1,1,1,1,1,0,0,0,0,0])
token = Tokenizer()
token.fit_on_texts(docs)   # 단어 사전 생성
print(token.word_index)

x = token.texts_to_sequences(docs)   # 토큰화
print(x)

# 시퀀스 데이터를 딥러닝 모델에 넣기 전에 토큰의 길이를 동일하게 해야함
from tensorflow.keras.utils import pad_sequences
padded_x = pad_sequences(x, maxlen=5, padding='pre')   # 'post'
print('패딩결과:\n', padded_x)


In [ ]:
# 모델 처리
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Input, Flatten

word_size = len(token.word_index) + 1  # 가능 토큰 갯수 + 1

model = Sequential()
model.add(Input(shape = (5, )))
model.add(Embedding(input_dim=word_size, output_dim=8))  # output_dim=8 : 각 단어를 8차원 실수 벡터로 표현
model.add(LSTM(32, activation='tanh'))
# model.add(Flatten())   # 출력은 이미 2D이므로 필요 없음
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
print(model.summary())

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(x=padded_x, y=labels, epochs=20, verbose=1)
print('정확도 : ', model.evaluate(padded_x, labels)[1])

print('예측 : ', np.where(model.predict(padded_x) > 0.5, 1, 0).ravel())